# Flappy Bird Q-Learning Project

This ipynb file is an alternative version of us presenting the project. It has the main implementation for the Flappy Bird reinforcement learning project. The goal is to train a tabular Q-learning agent to play a simplified Flappy Bird environment.

The project uses two configurations:

- **Baseline**: normal pipe speed
- **Fast Pipes**: harder version with faster pipe movement

The Narrow Gap experiment was removed from the final version to keep the project focused and easier to explain.

## 1. Project idea

The bird has two possible actions: **do nothing** or **flap**. The agent observes the current game state, chooses an action, receives a reward, and updates a Q-table. Over many episodes, the Q-table learns which action is better in each situation.

The visible score is the number of pipe pairs passed. The training reward is separate and is used only for learning:

- `+1` for surviving a frame
- `+5` for passing a pipe
- `-100` for crashing

In [ ]:
# Basic imports used in the notebook
import os
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

# Create project folders if they do not exist
os.makedirs("src", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 2. Q-learning agent

The agent stores a Q-table. Each state maps to two action values:

- action `0`: do nothing
- action `1`: flap

During training the agent uses epsilon-greedy exploration. This means it sometimes tries random actions, but later it mostly uses the best action it has learned.

In [ ]:
%%writefile src/agent.py
"""
Tabular Q-Learning agent for Flappy Bird.
"""

import numpy as np
import pickle
import os
from collections import defaultdict


class QLearningAgent:
    def __init__(
        self,
        alpha: float = 0.1,
        gamma: float = 0.99,
        epsilon: float = 1.0,
        epsilon_min: float = 0.01,
        epsilon_decay: float = 0.001,
        n_actions: int = 2,
    ):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.n_actions = n_actions

        # Q-table: state -> values for actions [no-op, flap]
        self.q_table = defaultdict(lambda: np.zeros(n_actions))

    def choose_action(self, state: tuple) -> int:
        """Epsilon-greedy action selection used during training."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[state]))

    def choose_action_greedy(self, state: tuple) -> int:
        """Always choose the best learned action. Used for evaluation/demo."""
        return int(np.argmax(self.q_table[state]))

    def update(self, state: tuple, action: int, reward: float, next_state: tuple, done: bool):
        """Single Q-learning update step."""
        target = reward
        if not done:
            target += self.gamma * np.max(self.q_table[next_state])

        td_error = target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error

    def decay_epsilon(self):
        """Reduce exploration after each episode."""
        self.epsilon = max(self.epsilon_min, self.epsilon - self.epsilon_decay)

    def save(self, path: str):
        data = {
            "q_table": dict(self.q_table),
            "alpha": self.alpha,
            "gamma": self.gamma,
            "epsilon": self.epsilon,
            "epsilon_min": self.epsilon_min,
            "epsilon_decay": self.epsilon_decay,
            "n_actions": self.n_actions,
        }
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(data, f)
        print(f"[Agent] Saved to {path}")

    @classmethod
    def load(cls, path: str) -> "QLearningAgent":
        with open(path, "rb") as f:
            data = pickle.load(f)

        agent = cls(
            alpha=data["alpha"],
            gamma=data["gamma"],
            epsilon=data["epsilon"],
            epsilon_min=data["epsilon_min"],
            epsilon_decay=data["epsilon_decay"],
            n_actions=data["n_actions"],
        )
        agent.q_table = defaultdict(lambda: np.zeros(agent.n_actions), data["q_table"])
        return agent

    def q_table_size(self) -> int:
        return len(self.q_table)

## 3. Flappy Bird environment

The environment is a simplified MDP version of Flappy Bird. The state has three parts:

- `dx`: horizontal distance to the next pipe
- `dy`: vertical distance from the bird to the gap center
- `vy`: bird vertical velocity

These values are discretized into bins so they can be used in a tabular Q-table.

In [ ]:
%%writefile src/flappy_env.py
"""
Flappy Bird Environment modeled as a Markov Decision Process (MDP).
Headless version for fast reinforcement learning training.
"""

import numpy as np
import random

# Game constants
SCREEN_W = 288
SCREEN_H = 512

GRAVITY = 1.0
FLAP_V = -9.0
MAX_V = 10.0

PIPE_VX = -4.0
PIPE_GAP = 150          # vertical gap between top and bottom pipe
PIPE_SPACING = 220      # horizontal distance between pipe pairs
MAX_GAP_JUMP = 120      # maximum vertical change between consecutive gaps

PIPE_W = 52
BIRD_X = 60
BIRD_W = 34
BIRD_H = 24


class FlappyBirdEnv:
    """
    Headless Flappy Bird environment.

    Actions:
        0 - do nothing
        1 - flap

    Reward:
        +1   each frame alive
        +5   when passing a pipe
        -100 on collision
    """

    DX_BINS = np.linspace(0, SCREEN_W, 20)
    DY_BINS = np.linspace(-SCREEN_H // 2, SCREEN_H // 2, 20)
    VY_BINS = np.linspace(-15, 15, 10)

    def __init__(self, pipe_gap: int = PIPE_GAP, pipe_speed: float = PIPE_VX):
        self.pipe_gap = pipe_gap
        self.pipe_speed = pipe_speed
        self.reset()

    def reset(self):
        self.bird_y = SCREEN_H // 2
        self.bird_vy = 0.0
        self.score = 0
        self.frame = 0
        self.done = False

        min_y = 70
        max_y = SCREEN_H - 70 - self.pipe_gap
        self.last_gap_y = random.randint(min_y, max_y)

        self.pipes = [
            self._new_pipe(SCREEN_W + 50),
            self._new_pipe(SCREEN_W + 50 + PIPE_SPACING),
        ]
        return self._get_state()

    def step(self, action: int):
        assert not self.done, "Call reset() before step()."

        if action == 1:
            self.bird_vy = FLAP_V

        self.bird_vy = min(self.bird_vy + GRAVITY, MAX_V)
        self.bird_y += self.bird_vy
        self.frame += 1

        reward = 1.0

        for pipe in self.pipes:
            pipe["x"] += self.pipe_speed

        for i, pipe in enumerate(self.pipes):
            if pipe["x"] + PIPE_W < 0:
                other = self.pipes[1 - i]
                self.pipes[i] = self._new_pipe(other["x"] + PIPE_SPACING)

        for pipe in self.pipes:
            if not pipe["passed"] and pipe["x"] + PIPE_W < BIRD_X:
                pipe["passed"] = True
                self.score += 1
                reward += 5.0

        if self._collision():
            self.done = True
            reward = -100.0

        return self._get_state(), reward, self.done, {"score": self.score}

    def get_state_continuous(self):
        return self._raw_state()

    def _new_pipe(self, x: float):
        min_y = 70
        max_y = SCREEN_H - 70 - self.pipe_gap

        low = max(min_y, self.last_gap_y - MAX_GAP_JUMP)
        high = min(max_y, self.last_gap_y + MAX_GAP_JUMP)
        gap_y = random.randint(low, high)

        self.last_gap_y = gap_y
        return {"x": x, "gap_y": gap_y, "passed": False}

    def _nearest_pipe(self):
        candidates = [p for p in self.pipes if p["x"] + PIPE_W >= BIRD_X]
        if not candidates:
            candidates = self.pipes
        return min(candidates, key=lambda p: p["x"])

    def _raw_state(self):
        pipe = self._nearest_pipe()
        dx = pipe["x"] - BIRD_X
        gap_center = pipe["gap_y"] + self.pipe_gap // 2
        dy = self.bird_y - gap_center
        vy = self.bird_vy
        return dx, dy, vy

    def _get_state(self):
        dx, dy, vy = self._raw_state()
        dx_d = int(np.digitize(dx, self.DX_BINS))
        dy_d = int(np.digitize(dy, self.DY_BINS))
        vy_d = int(np.digitize(vy, self.VY_BINS))
        return dx_d, dy_d, vy_d

    def _collision(self):
        if self.bird_y - BIRD_H // 2 <= 0:
            return True
        if self.bird_y + BIRD_H // 2 >= SCREEN_H:
            return True

        for pipe in self.pipes:
            bird_left = BIRD_X - BIRD_W // 2
            bird_right = BIRD_X + BIRD_W // 2
            bird_top = self.bird_y - BIRD_H // 2
            bird_bottom = self.bird_y + BIRD_H // 2

            pipe_left = pipe["x"]
            pipe_right = pipe["x"] + PIPE_W
            pipe_gap_top = pipe["gap_y"]
            pipe_gap_bottom = pipe["gap_y"] + self.pipe_gap

            if bird_right > pipe_left and bird_left < pipe_right:
                if bird_top < pipe_gap_top or bird_bottom > pipe_gap_bottom:
                    return True

        return False

## 4. Training and evaluation

The training file runs the Q-learning loop. It trains two agents:

- Baseline
- Fast Pipes

Each trained agent is saved as a `.pkl` file in the `models` folder. The `.pkl` file stores the learned Q-table and the agent settings.

In [ ]:
%%writefile src/train.py
"""
Training script for the Flappy Bird Q-Learning agent.
Runs Baseline and Fast Pipes experiments.
"""

import sys
import os
import json
import time

import numpy as np

sys.path.insert(0, os.path.dirname(__file__))

from flappy_env import FlappyBirdEnv, PIPE_GAP, PIPE_VX
from agent import QLearningAgent


RESULTS_DIR = os.path.join(os.path.dirname(__file__), "..", "results")
MODELS_DIR = os.path.join(os.path.dirname(__file__), "..", "models")


def train(env: FlappyBirdEnv, agent: QLearningAgent, n_episodes: int = 8000, log_interval: int = 500, label: str = "run") -> dict:
    scores = []
    rewards_total = []
    epsilons = []
    frames_alive = []

    t0 = time.time()

    for ep in range(1, n_episodes + 1):
        state = env.reset()
        done = False
        ep_reward = 0.0
        ep_frames = 0

        while not done:
            action = agent.choose_action(state)
            next_state, reward, done, info = env.step(action)
            agent.update(state, action, reward, next_state, done)
            state = next_state
            ep_reward += reward
            ep_frames += 1

        agent.decay_epsilon()

        scores.append(info["score"])
        rewards_total.append(ep_reward)
        epsilons.append(agent.epsilon)
        frames_alive.append(ep_frames)

        if ep % log_interval == 0:
            recent = scores[-log_interval:]
            elapsed = time.time() - t0
            print(
                f"[{label}] Ep {ep:>5}/{n_episodes}  "
                f"avg_score={np.mean(recent):.2f}  "
                f"max_score={max(recent)}  "
                f"epsilon={agent.epsilon:.3f}  "
                f"Q-states={agent.q_table_size()}  "
                f"time={elapsed:.1f}s"
            )

    return {
        "scores": scores,
        "rewards_total": rewards_total,
        "epsilons": epsilons,
        "frames_alive": frames_alive,
    }


def evaluate(env: FlappyBirdEnv, agent: QLearningAgent, n_episodes: int = 200) -> dict:
    scores = []
    frames = []

    for _ in range(n_episodes):
        state = env.reset()
        done = False
        fr = 0

        while not done:
            action = agent.choose_action_greedy(state)
            state, _, done, info = env.step(action)
            fr += 1

        scores.append(info["score"])
        frames.append(fr)

    return {
        "mean_score": float(np.mean(scores)),
        "max_score": int(max(scores)),
        "median_score": float(np.median(scores)),
        "std_score": float(np.std(scores)),
        "mean_frames": float(np.mean(frames)),
        "n_episodes": n_episodes,
    }


EXPERIMENTS = [
    {
        "label": "Baseline",
        "pipe_gap": PIPE_GAP,
        "pipe_speed": PIPE_VX,
        "n_episodes": 8000,
        "alpha": 0.1,
        "gamma": 0.99,
        "epsilon_decay": 0.0005,
    },
    {
        "label": "Fast_Pipes",
        "pipe_gap": PIPE_GAP,
        "pipe_speed": -6.0,
        "n_episodes": 8000,
        "alpha": 0.1,
        "gamma": 0.99,
        "epsilon_decay": 0.0005,
    },
]


def run_all():
    all_results = {}
    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.makedirs(MODELS_DIR, exist_ok=True)

    for cfg in EXPERIMENTS:
        label = cfg["label"]
        print("
" + "=" * 60)
        print(f"EXPERIMENT: {label}")
        print(f"pipe_gap={cfg['pipe_gap']}  pipe_speed={cfg['pipe_speed']}")
        print("=" * 60)

        env = FlappyBirdEnv(pipe_gap=cfg["pipe_gap"], pipe_speed=cfg["pipe_speed"])
        agent = QLearningAgent(
            alpha=cfg["alpha"],
            gamma=cfg["gamma"],
            epsilon=1.0,
            epsilon_min=0.01,
            epsilon_decay=cfg["epsilon_decay"],
        )

        metrics = train(env, agent, n_episodes=cfg["n_episodes"], label=label)
        eval_env = FlappyBirdEnv(pipe_gap=cfg["pipe_gap"], pipe_speed=cfg["pipe_speed"])
        eval_stats = evaluate(eval_env, agent, n_episodes=200)

        print(f"
[{label}] Eval -> {eval_stats}")

        agent.save(os.path.join(MODELS_DIR, f"agent_{label}.pkl"))

        result_data = {"train": metrics, "eval": eval_stats, "config": cfg}
        with open(os.path.join(RESULTS_DIR, f"metrics_{label}.json"), "w") as f:
            json.dump(result_data, f)

        all_results[label] = result_data

    with open(os.path.join(RESULTS_DIR, "all_results.json"), "w") as f:
        json.dump(all_results, f)

    print("
All experiments complete. Results saved.")
    return all_results


if __name__ == "__main__":
    run_all()

To train the models from the terminal, run:

```bash
python src/train.py
```

This generates:

- `models/agent_Baseline.pkl`
- `models/agent_Fast_Pipes.pkl`
- `results/all_results.json`
- `results/metrics_Baseline.json`
- `results/metrics_Fast_Pipes.json`

## 5. Analysis plots

The analysis script reads the saved JSON results and creates plots for learning curves, epsilon decay, score distribution, performance comparison, and survival time.

In [ ]:
%%writefile src/analyze.py
"""
Generate analysis plots for the Flappy Bird RL project.
Only Baseline and Fast Pipes are included.
"""

import json
import os
import sys

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


RESULTS_DIR = os.path.join(os.path.dirname(__file__), "..", "results")
MODELS_DIR = os.path.join(os.path.dirname(__file__), "..", "models")

EXPERIMENTS = {
    "Baseline": {
        "color": "#2196F3",
        "dark_color": "#1565C0",
        "label": "Baseline",
        "title": "Baseline",
        "gap": 150,
        "speed": -4.0,
    },
    "Fast_Pipes": {
        "color": "#4CAF50",
        "dark_color": "#1B5E20",
        "label": "Fast Pipes",
        "title": "Fast Pipes",
        "gap": 150,
        "speed": -6.0,
    },
}


def smooth(data, window=100):
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window) / window, mode="valid")


def load_results():
    path = os.path.join(RESULTS_DIR, "all_results.json")
    with open(path) as f:
        return json.load(f)


def fig1_learning_curves(results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

    for ax, (key, cfg) in zip(axes, EXPERIMENTS.items()):
        scores = results[key]["train"]["scores"]
        sm = smooth(scores, window=200)
        ep = np.arange(len(sm)) + 200

        ax.plot(ep, sm, color=cfg["color"], linewidth=2)
        ax.fill_between(ep, sm, alpha=0.15, color=cfg["color"])
        ax.set_title(cfg["title"], fontsize=11, fontweight="bold")
        ax.set_xlabel("Episode")
        ax.set_ylabel("Avg Score (200-episode window)")
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, len(scores))
        ax.set_ylim(bottom=0)

    plt.suptitle("Learning Curves - Q-Learning Agent on Flappy Bird", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(RESULTS_DIR, "fig1_learning_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")


def fig2_epsilon_decay(results):
    fig, ax = plt.subplots(figsize=(9, 4))

    for key, cfg in EXPERIMENTS.items():
        eps = results[key]["train"]["epsilons"]
        ax.plot(eps, color=cfg["color"], linewidth=2, label=cfg["label"])

    ax.set_xlabel("Episode")
    ax.set_ylabel("Epsilon (Exploration Rate)")
    ax.set_title("Epsilon Decay Over Training", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(RESULTS_DIR, "fig2_epsilon_decay.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")


def fig3_score_distribution(results):
    sys.path.insert(0, os.path.join(os.path.dirname(__file__)))
    from flappy_env import FlappyBirdEnv
    from agent import QLearningAgent

    all_scores = []
    labels_short = []

    for key, cfg in EXPERIMENTS.items():
        model_path = os.path.join(MODELS_DIR, f"agent_{key}.pkl")
        agent = QLearningAgent.load(model_path)
        env = FlappyBirdEnv(pipe_gap=cfg["gap"], pipe_speed=cfg["speed"])

        scores = []
        for _ in range(300):
            state = env.reset()
            done = False
            while not done:
                action = agent.choose_action_greedy(state)
                state, _, done, info = env.step(action)
            scores.append(info["score"])

        all_scores.append(scores)
        labels_short.append(cfg["label"])

    fig, ax = plt.subplots(figsize=(7, 5))
    bp = ax.boxplot(all_scores, patch_artist=True, widths=0.5, medianprops=dict(color="black", linewidth=2))

    for patch, cfg in zip(bp["boxes"], EXPERIMENTS.values()):
        patch.set_facecolor(cfg["color"])
        patch.set_alpha(0.7)

    ax.set_xticklabels(labels_short)
    ax.set_ylabel("Score (pipes passed)")
    ax.set_title("Evaluation Score Distribution (300 episodes per config)", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    path = os.path.join(RESULTS_DIR, "fig3_score_distribution.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")


def fig4_comparison_bar(results):
    keys = list(EXPERIMENTS.keys())
    labels = [EXPERIMENTS[k]["label"] for k in keys]
    means = [results[k]["eval"]["mean_score"] for k in keys]
    maxs = [results[k]["eval"]["max_score"] for k in keys]
    stds = [results[k]["eval"]["std_score"] for k in keys]

    x = np.arange(len(labels))
    width = 0.35
    colors_mean = [EXPERIMENTS[k]["color"] for k in keys]
    colors_max = [EXPERIMENTS[k]["dark_color"] for k in keys]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - width / 2, means, width, label="Mean Score", color=colors_mean, alpha=0.85, yerr=stds, capsize=5)
    bars2 = ax.bar(x + width / 2, maxs, width, label="Max Score", color=colors_max, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel("Score (pipes passed)")
    ax.set_title("Evaluation Performance Comparison", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f"{int(bar.get_height())}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    path = os.path.join(RESULTS_DIR, "fig4_comparison_bar.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")


def fig5_survival_frames(results):
    fig, ax = plt.subplots(figsize=(10, 4))

    for key, cfg in EXPERIMENTS.items():
        frames = results[key]["train"]["frames_alive"]
        sm = smooth(frames, window=200)
        ep = np.arange(len(sm)) + 200
        ax.plot(ep, sm, color=cfg["color"], linewidth=2, label=cfg["label"])

    ax.set_xlabel("Episode")
    ax.set_ylabel("Frames Survived (200-episode window)")
    ax.set_title("Survival Time During Training", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(RESULTS_DIR, "fig5_survival_frames.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")


def print_summary_table(results):
    print("
" + "=" * 70)
    print(f"{'Config':<20} {'Mean':>8} {'Max':>8} {'Median':>8} {'Std':>8}")
    print("-" * 70)

    for key, cfg in EXPERIMENTS.items():
        e = results[key]["eval"]
        print(f"{cfg['label']:<20} {e['mean_score']:>8.2f} {e['max_score']:>8} {e['median_score']:>8.1f} {e['std_score']:>8.2f}")

    print("=" * 70)


if __name__ == "__main__":
    results = load_results()
    fig1_learning_curves(results)
    fig2_epsilon_decay(results)
    fig3_score_distribution(results)
    fig4_comparison_bar(results)
    fig5_survival_frames(results)
    print_summary_table(results)
    print("
All figures saved.")

To generate the result figures from the terminal, run:

```bash
python src/analyze.py
```

This creates the plots inside the `results` folder.

## 6. Visual demo

The visual demo loads the saved `.pkl` model and shows the trained agent playing. The demo stops when the score reaches 150 and displays a win message.

In [ ]:
%%writefile src/play_visual.py
"""
Visual Flappy Bird demo for the trained Q-learning agent.
"""

import sys
import os
import argparse

sys.path.insert(0, os.path.dirname(__file__))

import pygame

from flappy_env import FlappyBirdEnv, SCREEN_W, SCREEN_H, PIPE_GAP, PIPE_VX, PIPE_W, BIRD_X
from agent import QLearningAgent

SKY = (113, 197, 207)
GROUND_COL = (222, 216, 149)
PIPE_COL = (115, 190, 75)
PIPE_DARK = (83, 154, 52)
BIRD_COL = (255, 215, 0)
BIRD_EYE = (30, 30, 30)
BIRD_BEAK = (255, 140, 0)
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (220, 50, 50)
GREEN = (80, 220, 120)

GROUND_H = 60
FPS_DEFAULT = 60
WIN_SCORE = 150


def draw_bird(surf, x, y, vy):
    r = 16
    pygame.draw.circle(surf, BIRD_COL, (int(x), int(y)), r)
    pygame.draw.circle(surf, (200, 170, 0), (int(x), int(y)), r, 2)
    wing_y = y + (4 if vy < 0 else 8)
    pygame.draw.ellipse(surf, (255, 180, 0), (int(x) - 8, int(wing_y) - 4, 16, 8))
    ex = int(x + r * 0.45)
    ey = int(y - r * 0.2)
    pygame.draw.circle(surf, WHITE, (ex, ey), 5)
    pygame.draw.circle(surf, BIRD_EYE, (ex + 1, ey), 3)
    bx = int(x + r * 0.8)
    by = int(y + 2)
    pygame.draw.polygon(surf, BIRD_BEAK, [(bx, by), (bx + 10, by - 3), (bx + 10, by + 3)])


def draw_pipe(surf, pipe, gap, screen_h, ground_h):
    x = int(pipe["x"])
    gap_y = pipe["gap_y"]
    cap_h = 18
    cap_w = PIPE_W + 8

    top_h = gap_y
    pygame.draw.rect(surf, PIPE_COL, (x, 0, PIPE_W, top_h))
    pygame.draw.rect(surf, PIPE_DARK, (x, 0, PIPE_W, top_h), 3)
    pygame.draw.rect(surf, PIPE_COL, (x - 4, top_h - cap_h, cap_w, cap_h))
    pygame.draw.rect(surf, PIPE_DARK, (x - 4, top_h - cap_h, cap_w, cap_h), 3)

    bot_y = gap_y + gap
    bot_h = screen_h - ground_h - bot_y
    pygame.draw.rect(surf, PIPE_COL, (x, bot_y, PIPE_W, bot_h))
    pygame.draw.rect(surf, PIPE_DARK, (x, bot_y, PIPE_W, bot_h), 3)
    pygame.draw.rect(surf, PIPE_COL, (x - 4, bot_y, cap_w, cap_h))
    pygame.draw.rect(surf, PIPE_DARK, (x - 4, bot_y, cap_w, cap_h), 3)


def draw_ground(surf, offset, screen_w, screen_h, ground_h):
    ground_y = screen_h - ground_h
    pygame.draw.rect(surf, GROUND_COL, (0, ground_y, screen_w, ground_h))
    pygame.draw.line(surf, (180, 170, 100), (0, ground_y), (screen_w, ground_y), 3)

    stripe_w = 40
    for i in range(-1, screen_w // stripe_w + 2):
        sx = i * stripe_w - int(offset) % stripe_w
        pygame.draw.line(surf, (200, 190, 120), (sx, ground_y + 10), (sx + 20, ground_y + 10), 2)


def draw_hud(surf, font_big, font_sm, score, best, episode, fps, speed, human_mode, alive):
    txt = font_big.render(str(score), True, WHITE)
    shadow = font_big.render(str(score), True, BLACK)
    cx = surf.get_width() // 2
    surf.blit(shadow, (cx - txt.get_width() // 2 + 2, 22))
    surf.blit(txt, (cx - txt.get_width() // 2, 20))

    lines = [
        f"Target:  {WIN_SCORE}",
        f"Best:    {best}",
        f"Episode: {episode}",
        f"FPS:     {fps}",
        f"Speed:   {speed}x",
        "",
        "SPACE - flap" if human_mode else "R - restart",
        "+/- speed",
        "Q - quit",
    ]

    pad = 8
    panel_w = 165
    panel_h = len(lines) * 20 + pad * 2
    panel = pygame.Surface((panel_w, panel_h), pygame.SRCALPHA)
    panel.fill((0, 0, 0, 140))
    surf.blit(panel, (8, 8))

    for i, line in enumerate(lines):
        col = (220, 220, 220)
        if "SPACE" in line or "R -" in line:
            col = (180, 255, 180)
        t = font_sm.render(line, True, col)
        surf.blit(t, (8 + pad, 8 + pad + i * 20))

    if not alive:
        msg = font_big.render("DEAD - press R", True, RED)
        surf.blit(msg, (cx - msg.get_width() // 2, SCREEN_H // 2 - 30))


def show_win_screen(screen, font_big, font_sm, score):
    screen.fill((0, 0, 0))
    title = font_big.render("YOU WON!", True, GREEN)
    score_text = font_sm.render(f"Score reached {score}", True, WHITE)
    quit_text = font_sm.render("Press Q or ESC to quit", True, WHITE)

    cx = SCREEN_W // 2
    cy = SCREEN_H // 2
    screen.blit(title, (cx - title.get_width() // 2, cy - 60))
    screen.blit(score_text, (cx - score_text.get_width() // 2, cy - 10))
    screen.blit(quit_text, (cx - quit_text.get_width() // 2, cy + 30))
    pygame.display.flip()

    waiting = True
    while waiting:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                waiting = False
            elif event.type == pygame.KEYDOWN and event.key in (pygame.K_q, pygame.K_ESCAPE):
                waiting = False


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="baseline", choices=["baseline", "fast_pipes"])
    parser.add_argument("--human", action="store_true")
    args = parser.parse_args()

    cfg_map = {
        "baseline": ("Baseline", PIPE_GAP, PIPE_VX),
        "fast_pipes": ("Fast_Pipes", PIPE_GAP, -6.0),
    }

    label, pipe_gap, pipe_speed = cfg_map[args.config]
    base = os.path.join(os.path.dirname(__file__), "..")
    model_path = os.path.join(base, "models", f"agent_{label}.pkl")

    if not args.human:
        if not os.path.exists(model_path):
            print(f"[ERROR] Model not found: {model_path}")
            print("Run: python src/train.py first.")
            sys.exit(1)
        agent = QLearningAgent.load(model_path)
        print(f"[Info] Loaded: {label} | Q-states={agent.q_table_size():,}")
    else:
        agent = None
        print("[Info] Human mode - SPACE to flap.")

    pygame.init()
    screen = pygame.display.set_mode((SCREEN_W, SCREEN_H))
    pygame.display.set_caption(f"Flappy Bird Q-Learning | AI - {label}")

    font_big = pygame.font.SysFont("Arial", 36, bold=True)
    font_sm = pygame.font.SysFont("Arial", 15)
    clock = pygame.time.Clock()

    env = FlappyBirdEnv(pipe_gap=pipe_gap, pipe_speed=pipe_speed)
    state = env.reset()
    done = False
    best_score = 0
    episode = 1
    speed_mult = 1
    ground_off = 0.0
    info = {"score": 0}

    running = True
    while running:
        human_flap = False

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                elif event.key == pygame.K_r:
                    state = env.reset()
                    done = False
                    episode += 1
                    info = {"score": 0}
                elif event.key in (pygame.K_EQUALS, pygame.K_PLUS, pygame.K_KP_PLUS, pygame.K_UP):
                    speed_mult = min(speed_mult + 1, 8)
                elif event.key in (pygame.K_MINUS, pygame.K_KP_MINUS, pygame.K_DOWN):
                    speed_mult = max(speed_mult - 1, 1)
                elif event.key == pygame.K_SPACE and args.human:
                    human_flap = True
            elif event.type == pygame.MOUSEBUTTONDOWN and args.human:
                human_flap = True

        for _ in range(speed_mult):
            if done:
                break

            if args.human:
                action = 1 if human_flap else 0
            else:
                action = agent.choose_action_greedy(state)

            state, reward, done, info = env.step(action)

            if info["score"] >= WIN_SCORE:
                best_score = max(best_score, info["score"])
                show_win_screen(screen, font_big, font_sm, info["score"])
                running = False
                break

        if done:
            best_score = max(best_score, info["score"])

        if done and not args.human and running:
            pygame.time.wait(400)
            state = env.reset()
            done = False
            episode += 1
            info = {"score": 0}

        ground_off -= abs(pipe_speed) * speed_mult
        if ground_off < -40:
            ground_off += 40

        if not running:
            break

        screen.fill(SKY)

        for cx, cy in [(60, 80), (160, 50), (240, 100)]:
            for dx, dy, r in [(-18, 0, 18), (0, -10, 22), (18, 0, 18), (0, 8, 16)]:
                pygame.draw.circle(screen, WHITE, (cx + dx, cy + dy), r)

        for pipe in env.pipes:
            draw_pipe(screen, pipe, env.pipe_gap, SCREEN_H, GROUND_H)

        draw_ground(screen, ground_off, SCREEN_W, SCREEN_H, GROUND_H)
        draw_bird(screen, BIRD_X, env.bird_y, env.bird_vy)
        draw_hud(screen, font_big, font_sm, info["score"], best_score, episode, int(clock.get_fps()), speed_mult, args.human, not done)

        pygame.display.flip()
        clock.tick(FPS_DEFAULT)

    pygame.quit()
    print(f"[Done] Best: {best_score} | Episodes: {episode}")


if __name__ == "__main__":
    main()

To run the visual demo:

```bash
python src/play_visual.py
```

For the faster pipe version:

```bash
python src/play_visual.py --config fast_pipes
```

The demo uses the saved `.pkl` model. If the model file is missing, run training first.

## 7. Results summary

The final comparison showed that the Baseline agent performed better than the Fast Pipes agent. The faster pipe speed made the game much harder because the agent had less time to react.

From the final evaluation:

| Configuration | Mean score | Max score | Median score |
|---|---:|---:|---:|
| Baseline | 1010.77 | 7579 | 617.5 |
| Fast Pipes | 30.54 | 238 | 13.0 |

The final demo uses the Baseline agent because it reached the 150-score win condition more reliably.

In [ ]:
# Optional: display saved result figures if they exist
from IPython.display import Image, display

figure_paths = [
    "results/fig1_learning_curves.png",
    "results/fig4_comparison_bar.png",
    "results/fig5_survival_frames.png",
]

for path in figure_paths:
    if os.path.exists(path):
        display(Image(filename=path))
    else:
        print(f"Missing: {path} - run python src/analyze.py after training.")

## 8. Conclusion

The project demonstrates reinforcement learning through a Flappy Bird game. The Q-learning agent starts with mostly random actions and gradually learns better behavior from rewards and penalties. The Baseline setting achieved the best performance, while the Fast Pipes setting showed that increasing speed makes the task harder for a tabular Q-learning agent.